# TerraWatch Milestone 4 — Qwen3-VL Verification Audit & Interactive Evidence Review

This notebook replays the **120-pair (147-claim) stratified verification study** comparing 5 claim-checking regimes on fixed full-data BIT masks (`D03_bit_lr3e4`) and RSCaMa captions (`C06_rscama_frozen_clip`), visually inspects all **8 authentic end-to-end traces**, and launches the interactive **TerraWatch Evidence Review UI** (`src/app/server.py`).

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

CANDIDATES = [Path.cwd(), Path.cwd().parent, Path("/content/TerraWatch")]
REPO = next((p for p in CANDIDATES if (p / "src/app/server.py").is_file()), None)
if REPO is None:
    REPO = Path("/content/TerraWatch")
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/shoaib2000857/Group-7-DS-and-AI-Lab-Project.git", str(REPO)],
        check=True,
    )

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

VER_DIR = REPO / "docs/milestone4/evidence/full_data_m4/verification"
TRACES_DIR = VER_DIR / "e2e_traces"
summary = json.loads((VER_DIR / "verification_comparison_summary.json").read_text(encoding="utf-8"))
print(f"Stratified audit pairs: {summary['num_audit_pairs']} | Atomic claims: {summary['num_claims']}")
print(f"Referee model: {summary['referee_model']}")

## 1. Verification Regime Comparison (Fixed Full-Data Masks & Claims)

In [ ]:
comp = summary["settings_comparison"]
print(f"{'Setting':<40} {'Prec':>7} {'Rec':>7} {'AccCov':>7} {'Defer':>7} {'Reject':>7} {'ContraCatch':>12}")
print("-" * 92)
for k, v in comp.items():
    print(
        f"{k:<40} {v['proxy_supported_precision_strict']*100:>6.2f}% "
        f"{v['proxy_supported_recall']*100:>6.2f}% {v['accept_rate']*100:>6.2f}% "
        f"{v['defer_rate']*100:>6.2f}% {v['reject_rate']*100:>6.2f}% "
        f"{v['proxy_contradicted_catch_rate']*100:>11.2f}%"
    )

## 2. Inspect All 8 Authentic End-to-End Pipeline Traces

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

trace_index = json.loads((TRACES_DIR / "trace_index.json").read_text(encoding="utf-8"))
print(f"Loaded {len(trace_index)} authentic E2E traces:")
for item in trace_index:
    rep = json.loads((TRACES_DIR / item["folder"] / "report.json").read_text(encoding="utf-8"))
    c0 = rep["claims"][0] if rep["claims"] else {}
    outcome = c0.get("decision", {}).get("outcome", "n/a").upper()
    print(f"  - {item['pair_id']:<18} [{outcome:<6}] ({item['category']}) | caption: {rep['generated_caption']!r}")

# Plot first 4 diverse traces inline
fig, axes = plt.subplots(4, 5, figsize=(14, 11))
titles = ["T1", "T2", "BIT Overlay + Box", "Crop T1 (32px+)", "Crop T2 (32px+)"]
for r_i, item in enumerate(trace_index[:4]):
    folder = TRACES_DIR / item["folder"]
    rep = json.loads((folder / "report.json").read_text(encoding="utf-8"))
    c0 = rep["claims"][0]
    imgs = [
        Image.open(folder / "T1.png"),
        Image.open(folder / "T2.png"),
        Image.open(folder / "mask_overlay.png"),
        Image.open(folder / c0["evidence"]["crop_paths"][0]),
        Image.open(folder / c0["evidence"]["crop_paths"][1]),
    ]
    for c_i, im in enumerate(imgs):
        axes[r_i][c_i].imshow(im)
        axes[r_i][c_i].axis("off")
        if r_i == 0:
            axes[r_i][c_i].set_title(titles[c_i], fontsize=10, fontweight="bold")
    axes[r_i][0].set_ylabel(f"{item['pair_id']}\n[{c0['decision']['outcome'].upper()}]", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Run Full Verification Audit (GPU) or Launch Interactive Evidence Review UI

To explore all 8 real traces interactively in your browser — with immutable model outputs, zoomed crops, Qwen3-VL probability bars, disagreement triggers, and append-only SQLite audit persistence — run:

```bash
python -m src.app.server docs/milestone4/evidence/full_data_m4/verification/e2e_traces --port 8765
```

In [ ]:
RUN_AUDIT = False
if RUN_AUDIT:
    subprocess.run([
        sys.executable, "scripts/run_verification_audit.py",
        "--data-dir", "data/processed/levir_mci",
        "--detector-arch", "bit_resnet18",
        "--detector-ckpt", "outputs/detector_runs/D03_bit_lr3e4/best.pt",
        "--captioner-ckpt", "outputs/captioner_runs/C06_rscama_frozen_clip/best.pth",
        "--captioner-source-dir", "/content/RSCaMa",
        "--bit-source-dir", "/content/BIT_CD",
        "--out-dir", "outputs/verification_audit",
        "--num-audit-pairs", "120",
    ], check=True)

# Uncomment to launch the local Evidence Review UI server on port 8765:
# !python -m src.app.server docs/milestone4/evidence/full_data_m4/verification/e2e_traces --port 8765